In [1]:
import numpy as np
import periodictable
import sys
sys.path.append('..')
import units

In [ ]:
# change mass unit from kg to be composed of eV AA and ps
E = m  c^2
m = E/c^2 
1 kg = 1/(299792458)^2 J m^-2 s^2
     = 1/89875517873681764 J m^-2 s^2
     = 1.112650056053618432174089964848e-17 J m^-2 s^2
     = 1.112650056053618432174089964848e-17 * 6.24150913e18 eV m^-2 s^2
     = 1.112650056053618432174089964848e-17 * 6.24150913e18 * (1e10)^-2 * (1e12)^2 eV AA^-2 ps^2
     = 694461.548335367 eV AA^-2 ps^2
    
1 atomic mass (Da) = 1.66053906660e-27 kg
                   = 1.1531805312624011e-21 eV AA^-2 ps^2

In [24]:
print(1.112650056053618432174089964848e-17 * 6.24150913e18 * (1e10)**-2 * (1e12)**2)

694461.548335367


In [26]:
print(694461.548335367 * 1.66053906660e-27)

1.1531805312624011e-21


In [2]:
units.Dalton

1.1531805312624011e-21

In [15]:
# structure.py

def read_xyz_file(xyz_file):
    xyz = open(xyz_file).readlines()
    
    structure = {}
    structure['n_atoms'] = int(xyz[0].split()[0])
    try:
        structure['total_energy'] = float(xyz[1].split()[0])
    except:
        pass
    
    element_list = []
    mass_list = []
    str_x_list = []
    str_y_list = []
    str_z_list = []
    str_vx_list = []
    str_vy_list = []
    str_vz_list = []
    str_fx_list = []
    str_fy_list = []
    str_fz_list = []
    for i_atom in range(structure['n_atoms']):
        element_list.append(str(xyz[i_atom+2].split()[0]))
        str_x_list.append(float(xyz[i_atom+2].split()[1]))
        str_y_list.append(float(xyz[i_atom+2].split()[2]))
        str_z_list.append(float(xyz[i_atom+2].split()[3]))
        try:
            str_vx_list.append(float(xyz[i_atom+2].split()[4]))
            str_vy_list.append(float(xyz[i_atom+2].split()[5]))
            str_vz_list.append(float(xyz[i_atom+2].split()[6]))
        except:
            str_vx_list.append(0.0)
            str_vy_list.append(0.0)
            str_vz_list.append(0.0)
            pass
            
        try:
            str_fx_list.append(float(xyz[i_atom+2].split()[7]))
            str_fy_list.append(float(xyz[i_atom+2].split()[8]))
            str_fz_list.append(float(xyz[i_atom+2].split()[9]))
        except:
            pass
            
    structure['elements'] = element_list
    for i_ele in element_list:
        mass_list.append(periodictable.elements.symbol(i_ele).mass)
    structure['masses'] = np.array(mass_list) * units.Dalton
    
    str_x_list = np.array(str_x_list)
    str_y_list = np.array(str_y_list)
    str_z_list = np.array(str_z_list)
    structure['coordinates'] = np.array([str_x_list, str_y_list, str_z_list]).T
    
    str_vx_list = np.array(str_vx_list)
    str_vy_list = np.array(str_vy_list)
    str_vz_list = np.array(str_vz_list)
    structure['velocities'] = np.array([str_vx_list, str_vy_list, str_vz_list]).T
    
    acceleration_list = []
    if len(str_fx_list) > 0:
        str_fx_list = np.array(str_fx_list)
        str_fy_list = np.array(str_fy_list)
        str_fz_list = np.array(str_fz_list)
        structure['forces'] = np.array([str_fx_list, str_fy_list, str_fz_list]).T
        structure['accelerations'] = np.array([structure['forces'][i_atom]/structure['masses'][i_atom] \
                                          for i_atom in range(len(structure['forces']))])
    
    return structure

def write_xyz_file(xyz_file, structure_dict):
    xyz_file.write(str(next_structure['n_atoms'])+'\n')
    xyz_file.write(str(next_structure['total_energy'])+'\n')
    for i_atom in range(next_structure['n_atoms']):
        xyz_file.write(next_structure['element'][i_atom]+'    ')
        for i in range(3):
            xyz_file.write(str(next_structure['coordinates'][i_atom][i])+'    ')
        for i in range(3):
            xyz_file.write(str(next_structure['velocities'][i_atom][i])+'    ')
        for i in range(3):
            xyz_file.write(str(next_structure['forces'][i_atom][i])+'    ')
        xyz_file.write('\n')

In [17]:
print(read_xyz_file('MoREST.str_new'))

{'n_atoms': 2, 'total_energy': 342.2, 'elements': ['Al', 'F'], 'masses': array([3.11145843e-20, 2.19085887e-20]), 'coordinates': array([[0.1, 0.2, 0.3],
       [1.1, 1.2, 1.3]]), 'velocities': array([[ 1.,  2.,  3.],
       [-1., -2., -3.]]), 'forces': array([[ 10.,  20.,  30.],
       [-10., -20., -30.]]), 'accelerations': array([[ 3.21392691e+20,  6.42785383e+20,  9.64178074e+20],
       [-4.56441998e+20, -9.12883996e+20, -1.36932599e+21]])}


In [4]:
# potential.py

class ml_potential:
    def __init__(self):
        return None
        
    def read_potential_force(self, structure):
        return np.random.random_sample(), np.random.rand(2,3)

In [ ]:
# phase_space_sampling.py

class velocity_Verlet:
    '''
    This class implements velocity Verlet algorithm to do microcanonical ensemble (NVE MD) sampling.
    MoREST.traj records the trajectory in an extended xyz format:
    ------------------------------------
    n_atoms
    total energy
    element1 x1 y1 z1 velocity_x1 velocity_y1 velocity_z1 force_x1 force_y1 force_z1
    ...
    ------------------------------------
    MoREST.str records the initial xyz structure of the system
    MoREST.str_new records the current xyz structure of the system
    '''
    
    def __init__(self, sampling_parameters, md_parameters):
        #self.md_parameters = np.load('MoREST_md_parameters.npy',allow_pickle=True).item()
        self.sampling_parameters = sampling_parameters
        self.md_parameters = md_parameters
        if self.sampling_parameters['sampling_restart']:
            self.__log_traj = open('MoREST.traj','a')
            self.system_structure = structure_xyz().read_xyz_file('MoREST.str_new')
            self.structure = structure_xyz().read_xyz_file('MoREST.str_new')
        else:
            self.__log_traj = open('MoREST.traj','w')
            self.system_structure = structure_xyz().read_xyz_file('MoREST.str')
            self.structure = structure_xyz().read_xyz_file('MoREST.str')
            self.structure['total_energy'], self.structure['forces'] = ml_potential().read_potential_force(self.structure['coordinates'])
            self.structure['accelerations'] = np.array([self.structure['forces'][i_atom]/self.structure['masses'][i_atom] \
                                              for i_atom in range(len(self.structure['forces']))])
            
            write_xyz_file(self.__log_traj, self.structure)
        
    def generate(self):
        time_step = self.md_parameters['time_step']
        self.next_structure = {}
        
        self.next_structure['n_atoms'] = self.structure['n_atoms']
        self.next_structure['elements'] = self.structure['elements']
        self.next_structure['masses'] = self.structure['masses']
        self.next_structure['coordinates'] = self.structure['coordinates'] \
                                           + self.structure['velocities'] * time_step \
                                           + 0.5 * self.structure['accelerations'] * time_step**2
        self.next_structure['total_energy'], self.next_structure['forces'] = \
                                           ml_potential().read_potential_force(self.next_structure['coordinates'])
        self.next_structure['accelerations'] = np.array([self.next_structure['forces'][i_atom]/self.next_structure['masses'][i_atom] \
                                              for i_atom in range(len(self.next_structure['forces']))])
        self.next_structure['velocities'] = self.structure['velocities'] \ 
                                          + 0.5 * (self.structure['accelerations'] + self.next_structure['accelerations']) * time_step
        
        self.structure = self.next_structure
        str_new = open('MoREST.str_new','w')
        write_xyz_file(str_new, self.structure)
        write_xyz_file(self.__log_traj, self.structure)
        
        return None

In [5]:
periodictable.elements.symbol('Al').mass

26.981538